# 2장 1강: API를 이용한 데이터 수집 — 실습문제


## 실습 목표

- API의 역할과 Client·Server의 요청·응답 흐름을 설명할 수 있다.
- `requests.get()`으로 페이지 조건을 전달하고 URL·상태 코드·응답 형식을 확인할 수 있다.
- `response.json()`으로 JSON을 파싱하고 명언 본문·작성자·태그를 추출할 수 있다.
- `page`와 `has_next`를 이용하여 여러 페이지를 순서대로 수집할 수 있다.
- 정상 종료한 수집 결과를 DataFrame과 CSV로 저장하고 다시 읽어 기본 구조를 확인할 수 있다.
- 요청 성공, JSON 파싱 성공, 데이터 추출 성공, 전체 페이지 수집 완료를 구분할 수 있다.

## 실습 환경 / 데이터

- 환경: Python 3, Jupyter Notebook 또는 Google Colab, 인터넷 연결
- 사용 라이브러리: `requests`, `pandas`, `time`(Python 기본 모듈)
- 기준 교안: **2장 1강: API를 이용한 데이터 수집**
- 수집 대상: [Quotes to Scrape API](https://quotes.toscrape.com/api/quotes?page=1)
- 기본 URL: `https://quotes.toscrape.com/api/quotes`
- 요청 조건: `params={"page": 페이지번호}`
- 입력 CSV는 필요하지 않습니다. 실제 API에 요청하여 받은 데이터로 결과 파일을 만듭니다.

교안의 사이트·응답 구조·기능을 사용합니다. 별도의 인증, 비동기 요청, 자동 재시도, HTML 파싱, DB 저장, 결측·중복 처리 기능을 추가로 요구하지 않습니다.

| 문제 | 수집 범위 | 주요 내용 | 결과물 |
|---|---|---|---|
| 필수 1 | 1페이지 | 요청·응답 확인, JSON 파싱, 필드 추출 | `single_records`, 확인표 |
| 필수 2 | 1페이지부터 마지막 페이지까지 | 페이지네이션, 오류 시 중단, 정상 종료 후 저장 | `quotes_raw.csv` |
| 과제 1 | 2페이지 한 페이지만 | 필수에서 배운 요청·추출·저장을 독립 적용 | `quotes_page2.csv` |

**공통 데이터 구조**

| 최종 컬럼 | 의미 | 값의 위치·구성 |
|---|---|---|
| `text` | 명언 본문 | 각 명언의 `text` |
| `author` | 작성자 이름 | 각 명언의 `author` → `name` |
| `tags` | 관련 태그 | 각 명언의 `tags` 목록을 쉼표와 공백으로 연결 |
| `source_url` | 수집 출처 | 해당 응답의 `response.url` |

한 행은 명언 하나입니다. 응답의 `page`와 `has_next`는 수집 진행을 판단하는 정보이며, 이번 CSV의 네 컬럼에는 넣지 않습니다.


## 시작 전에 읽을 기초 개념

1장에서는 준비된 HTML·JSON에서 필요한 값을 꺼내 표와 파일로 만들었습니다. 이번에는 **Python이 서버에 요청해 받은 응답**을 사용합니다. 값을 정리하는 방법은 앞서 배운 방법과 연결되고, 그 앞에 요청·응답 확인 과정이 추가됩니다.

### 용어를 이번 실습에 연결하기

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습의 예 |
|---|---|---|
| API | 프로그램이 정해진 방식으로 다른 프로그램의 데이터·기능을 이용하는 통로 | 페이지 번호를 보내 명언 데이터를 받는 기능 |
| Client | 요청을 보내는 프로그램 | 노트북에서 실행하는 Python 코드 |
| Server | 요청을 처리하고 응답하는 시스템 | Quotes to Scrape 서버 |
| HTTP | 웹에서 요청과 응답을 주고받는 규칙 | Python이 요청하고 서버가 상태·데이터를 반환 |
| GET | 서버의 데이터를 조회하는 요청 방식 | 지정한 페이지의 명언 조회 |
| URL | 요청할 곳을 나타내는 주소 | API_URL의 기본 주소와 페이지 조건 |
| Query Parameter | URL에 전달하는 요청 조건 | page=1 |
| Response | 서버에서 돌아온 응답 | 상태 코드·헤더·본문 등을 가진 response |
| Status Code | HTTP 요청의 처리 결과를 나타내는 숫자 | 정상 처리의 200, 자원을 못 찾은 404 |
| Headers / Body | 응답에 관한 설명 / 실제 응답 내용 | Content-Type / 명언 JSON |
| Content-Type | 서버가 알려 주는 응답 내용의 형식 | application/json |
| JSON 파싱 | JSON 내용을 Python 값으로 해석하는 일 | response.json()으로 dict·list 등을 얻음 |
| Pagination | 많은 데이터를 페이지로 나누어 제공하는 방식 | page 번호를 바꾸어 차례로 요청 |
| has_next | 다음 페이지가 있는지 알려 주는 값 | True면 있음, False면 없음 |
| 예외(Exception) | 실행을 평소 순서대로 이어가기 어려운 오류 상황 | 요청 실패, 없는 Key 접근 |
| Timeout | 통신을 무기한 기다리지 않도록 제한하는 설정·상황 | 요청 코드의 timeout=10 |
| 레코드 / 필드 | 명언 하나의 정보 묶음 / 그 묶음의 항목 | dict 하나 / text·author·tags·source_url |
| 원시 데이터 | 수집해 정리했지만 본격적인 품질 정제 전인 데이터 | quotes_raw.csv |

**API, JSON, CSV는 같은 말이 아닙니다.** API는 요청하는 방법·기능이고, JSON은 이 API가 응답 내용을 표현하는 형식이며, CSV는 추출 결과를 표 형태로 저장할 파일 형식입니다.

### 한 번의 요청이 지나가는 단계

| 순서 | 하는 일 | 작업 결과 | 아직 확인되지 않은 것 |
|---|---|---|---|
| 1 | API 주소와 페이지 조건 지정 | 요청 설정 | 서버가 응답할 수 있는지 |
| 2 | requests.get()으로 요청 | 응답 객체 | HTTP 오류인지, 본문을 쓸 수 있는지 |
| 3 | 상태·형식 확인, raise_for_status() | HTTP 오류 확인 | JSON 파싱과 필드 추출 성공 |
| 4 | response.json() | Python dict·list 등의 값 | 필요한 Key와 명언이 있는지 |
| 5 | 명언별 필드를 꺼내 dict로 묶기 | 레코드 목록 | 모든 페이지를 수집했는지 |
| 6 | 필요하면 다음 페이지 요청 | 여러 페이지의 누적 목록 | 저장 결과가 읽히는지 |
| 7 | DataFrame → CSV → 다시 읽기 | 저장 결과와 기본 확인 출력 | 중복·결측 등 전체 품질 검증 |

상태 코드 200은 이 흐름 중 **HTTP 요청 처리 결과**입니다. 마지막 단계까지 모두 성공했다는 표시로 사용하지 않습니다.

### 1장에서 배운 내용과 연결하기

| 앞에서 사용한 방법 | 이번에 연결되는 방법 |
|---|---|
| 준비된 JSON 문자열에 json.loads() 적용 | 받은 HTTP 응답에 response.json() 적용 |
| 중첩 dict의 Key를 따라 값 선택 | quote["author"]["name"]으로 이름 추출 |
| 명언별 dict를 list에 append() | 페이지별 명언을 같은 목록에 누적 |
| 태그 list를 join()으로 연결 | 여러 태그를 tags 한 칸의 문자열로 정리 |
| 예시 출처 주소를 직접 기록 | 현재 응답의 response.url을 source_url에 기록 |
| DataFrame을 CSV로 저장하고 다시 읽기 | 전체 또는 지정한 한 페이지 결과를 각각 저장 |

### 코드를 읽을 때 먼저 구분할 것

| 표현 | 의미 |
|---|---|
| `params={"page": page}` | 왼쪽 page 글자는 서버에 보낼 조건 이름, 오른쪽 page 변수는 현재 번호 |
| `single_response` | 상태·헤더·본문 등을 가진 응답 객체 |
| `single_data` | 응답 본문을 JSON으로 해석한 값 |
| `single_quotes` | 해석한 값 안에서 꺼낸 이번 페이지의 명언 목록 |
| `single_records` | 필요한 네 필드로 새로 정리한 레코드 목록 |
| `try` / `except` / `else` | 우선 실행 / 지정한 오류 처리 / try에서 예외 없이 끝났을 때 실행 |
| `if` / `not` / `or` | 조건이면 실행 / 참·거짓을 반대로 판단 / 둘 중 하나라도 참인지 판단 |
| `while True` / `break` | 반복을 계속함 / 가장 가까운 반복문을 종료 |
| `page += 1` | page 값을 현재 값보다 1 크게 바꿈 |
| `raise ValueError(...)` | 정한 조건의 문제를 발견했음을 예외로 알림 |

코드의 들여쓰기는 **어느 블록 안에서 실행하는지**를 결정합니다. 단순히 보기 좋게 띄운 공간이 아닙니다. 특히 값 추출은 for 안, 페이지 이동은 while 안, 전체 결과 저장은 while 밖에 있다는 점을 구분하세요.


## 실습 준비

1. `requests`와 `pandas`가 설치된 Python 환경에서 아래 제공 셀을 실행하세요. 두 패키지를 찾을 수 없다는 오류가 나면 설치된 실습 환경을 준비한 뒤 진행하세요.
2. 아래 API 주소와 최종 컬럼을 확인하세요. 준비 셀은 아직 API를 호출하지 않습니다.
3. 필수 1 → 필수 2 → 과제 1 순서로 실행하세요. 각 문제의 결과 목록은 해당 문제 안에서 빈 List로 초기화합니다.
4. 사이트 응답은 실행 시점에 따라 달라질 수 있습니다. 고정된 숫자를 맞추는 대신 실제 페이지 정보·건수·컬럼을 기록하세요.

**실행 안내**

- 오류가 나면 상태 코드와 메시지를 확인하세요. 교안의 설명처럼 403·429 등 접근 제한 상황에서 요청을 무조건 반복하지 않습니다.
- 실제 오류를 만들기 위해 잘못된 주소로 요청하거나 과도하게 호출할 필요는 없습니다. 오류 구분 문제는 제시된 상황을 읽고 답합니다.
- 필수 2는 교안처럼 요청 사이에 0.5초 간격을 두고 최대 20페이지까지만 시도합니다. 정상 종료 기준은 `has_next=False`입니다.
- CSV는 현재 작업 폴더에 저장됩니다. 같은 이름으로 성공적으로 다시 저장하면 파일 내용이 갱신됩니다.
- 수집 도중 오류가 나면 이번 실행은 CSV 저장으로 진행하지 않습니다. 이전 실행의 같은 이름 파일이 남아 있을 수 있으므로, 파일 존재만으로 이번 실행 성공을 판단하지 마세요.


In [ ]:
import time
import requests
import pandas as pd

API_URL = "https://quotes.toscrape.com/api/quotes"
COLUMNS = ["text", "author", "tags", "source_url"]

print("API 주소:", API_URL)
print("최종 컬럼:", COLUMNS)
print("준비 완료: 아직 API 요청은 보내지 않았습니다.")


### 준비 코드 상세 해설

| 코드 | 역할 | 준비 셀 실행 직후 |
|---|---|---|
| `import time` | 대기 시간을 줄 도구 준비 | 이후 time.sleep() 사용 가능 |
| `import requests` | HTTP 요청 도구 준비 | 아직 요청은 보내지 않음 |
| `import pandas as pd` | 표·CSV 처리 도구를 pd라는 이름으로 준비 | 아직 DataFrame은 만들지 않음 |
| `API_URL = ...` | 페이지 조건을 제외한 기본 API 주소 보관 | str |
| `COLUMNS = [...]` | 저장할 네 필드의 이름·순서 보관 | 문자열 네 개를 담은 list |
| 세 print() | 준비한 주소·컬럼과 안내 문구 표시 | 서버 접속 결과가 아닌 설정 확인 |

API_URL과 COLUMNS를 대문자로 쓰는 것은 공통 설정임을 알아보기 쉽게 한 표기입니다. 변수 이름을 대문자로 썼다고 서버가 호출되거나 Python이 값 변경을 자동으로 금지하는 것은 아닙니다.

### 요청 주소를 부분별로 읽기

| URL의 부분 | 값 | 의미 |
|---|---|---|
| 통신 방식 | https | 서버와 통신하는 방식 |
| 도메인 | quotes.toscrape.com | 요청을 보낼 서버 주소 |
| 경로 | /api/quotes | 사용할 데이터 경로 |
| 조건 시작 표시 | ? | 뒤에 요청 조건이 이어짐 |
| Query Parameter | page=1 | 1페이지를 요청하는 조건 |

`requests.get(API_URL, params={"page": 1}, timeout=10)`에서 page는 서버로 보내는 조건이고, timeout은 Python 요청 도구가 기다림을 제한하는 설정입니다. timeout=10을 서버의 페이지 번호나 명언 개수로 해석하지 않습니다. 또한 전체 수집이 반드시 10초 안에 끝난다는 뜻도 아닙니다.

세 문제는 각자 새 결과 목록으로 시작합니다. 필수 2는 필수 1의 single_records에 이어 붙이는 코드가 아니라 1페이지부터 다시 수집합니다. 과제도 전체 수집 결과를 잘라 쓰는 대신 2페이지를 직접 요청하는 연습입니다.


---

## 필수 1. 한 페이지의 API 요청과 JSON 필드 추출

### 문제 1-1. 요청 성공과 실제 추출 결과 구분하기

#### 문제 설명

Quotes to Scrape API의 첫 번째 페이지를 요청하여 응답 상태를 확인하고, 명언을 공통된 네 필드로 정리하세요. 요청이 성공했는지와 실제 명언을 추출했는지를 구분하여 설명합니다.

#### 요구사항

1. `single_params = {"page": 1}`을 설정하고 `single_records = []`로 시작하세요.
2. `try` 안에서 `requests.get(API_URL, params=single_params, timeout=10)`을 호출하여 `single_response`에 저장하세요. 요청 URL, 상태 코드, `Content-Type`을 출력하고 `raise_for_status()`로 HTTP 오류를 확인하세요.
3. `single_response.json()`을 `single_data`에 저장하세요. JSON 파싱 오류는 `requests.exceptions.JSONDecodeError`, 요청 오류는 `requests.RequestException`으로 구분하여 출력하세요. JSON 파싱 오류를 먼저 처리하는 순서는 교안 예제와 같습니다.
4. 요청·파싱 성공 후 실행되는 `else`에서 `single_data["quotes"]`를 순회하세요. 본문·작성자 이름·쉼표로 연결한 태그·`single_response.url`을 네 Key의 Dictionary로 만들어 `single_records`에 추가하세요.
5. 현재 페이지, `has_next`, 응답 명언 수, 추출 건수를 출력하세요. `if single_records:`로 비어 있지 않을 때만 첫 레코드의 Key 목록·작성자·태그를 출력하고, 비어 있으면 명언이 없다는 안내를 출력하세요.
6. 실행 결과 확인표와 Q1~Q5에 답하세요. 요청이 실패했다면 성공한 값으로 채우지 말고 오류 상황을 적으세요.

#### 해석 질문

**Q1.** 이번 실습의 Client와 Server는 무엇이며, GET 요청은 어떤 목적으로 사용하나요?

**Q2.** 첫 페이지 요청 URL에서 도메인·경로·Query Parameter는 각각 무엇인가요? `params`와 `timeout=10`은 어떤 역할을 하나요?

**Q3.** `raise_for_status()`와 `response.json()`은 각각 무엇을 확인하거나 처리하나요? 상태 코드 200이면 명언 추출까지 성공한 것인가요?

**Q4.** 작성자 이름을 가져올 때 `author` 다음에 `name`까지 접근하는 이유는 무엇인가요? `source_url`에는 무엇을 저장하나요?

**Q5.** 다음 상황을 구분하세요: A. 200·JSON 파싱 성공·명언 10개, B. 200·JSON 파싱 성공·빈 명언 목록, C. 404, D. 200이지만 JSON으로 해석할 수 없는 응답.

#### 제출 결과

- API 요청·추출 코드와 출력 결과
- 실행 결과 확인표
- Q1~Q5 답변


### 작성 도움말 — 필수 1

요청 설정 → 요청·응답 확인 → JSON 파싱 → 명언 목록 선택 → 레코드 구성 → 결과 출력 순서로 작성하세요.

- response와 파싱한 data를 같은 것으로 생각하지 마세요. 상태 코드는 응답에서, 명언 목록은 파싱한 값에서 찾습니다.
- try 안의 요청·파싱과 else 안의 추출을 들여쓰기로 구분합니다. 오류가 나면 성공 결과를 임의로 채우지 않습니다.
- 첫 레코드를 `[0]`으로 꺼내기 전에 결과 목록이 비어 있지 않은지 확인하세요.
- 해석에서는 상태 코드, 파싱 여부, 실제 레코드 수를 각각 구분합니다. 한 페이지의 결과임을 함께 적습니다.


### 코드 작성란

아래 셀에 코드를 작성하고 실제 실행 결과를 남기세요.


In [ ]:
# 필수 1. 한 페이지의 API 요청과 JSON 필드 추출 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.


### 답변 작성란


| 확인 항목 | 실행 결과·오류 상황 | 해석 |
|---|---|---|
| 요청 URL |  |  |
| 상태 코드 / 응답 형식 |  |  |
| 현재 페이지 / 다음 페이지 존재 |  |  |
| 응답 명언 수 / 추출 건수 |  |  |
| 첫 레코드의 Key 목록 |  |  |
| 첫 작성자 / 첫 태그 |  |  |


**Q1. 이번 실습의 Client와 Server는 무엇이며, GET 요청은 어떤 목적으로 사용하나요?**

- 답변: 

**Q2. 첫 페이지 요청 URL에서 도메인·경로·Query Parameter는 각각 무엇인가요? `params`와 `timeout=10`은 어떤 역할을 하나요?**

- 답변: 

**Q3. `raise_for_status()`와 `response.json()`은 각각 무엇을 확인하거나 처리하나요? 상태 코드 200이면 명언 추출까지 성공한 것인가요?**

- 답변: 

**Q4. 작성자 이름을 가져올 때 `author` 다음에 `name`까지 접근하는 이유는 무엇인가요? `source_url`에는 무엇을 저장하나요?**

- 답변: 

**Q5. 다음 상황을 구분하세요: A. 200·JSON 파싱 성공·명언 10개, B. 200·JSON 파싱 성공·빈 명언 목록, C. 404, D. 200이지만 JSON으로 해석할 수 없는 응답.**

- 답변:


---

## 필수 2. 여러 페이지 수집과 CSV 저장

### 문제 2-1. 마지막 페이지까지 수집한 경우에만 결과 저장하기

#### 문제 설명

교안의 전체 수집 예제를 적용하여 1페이지부터 마지막 페이지까지 명언을 모으세요. `has_next`로 종료 여부를 판단하고, 중간 오류나 페이지 상한 초과를 전체 수집 완료로 처리하지 않아야 합니다.

#### 요구사항

1. `all_records = []`, `page = 1`, `max_pages = 20`으로 시작하세요. `try` 안에서 `while True`로 페이지를 반복하세요.
2. 각 요청 전에 `page > max_pages`이면 `RuntimeError`를 발생시키세요. 현재 페이지 조건과 `timeout=10`으로 요청하고, `raise_for_status()` 후 `.json()`을 사용하세요.
3. `data["quotes"]`가 List가 아니거나 비어 있으면 교안처럼 `ValueError`를 발생시키세요. 각 명언을 동일한 네 Key로 정리하고 현재 `response.url`을 출처로 넣어 `all_records`에 누적하세요.
4. 현재 요청 페이지·이번 페이지 건수·누적 건수를 출력하세요. 현재 페이지 명언을 추가한 뒤 `if not data["has_next"]: break`로 종료하세요. 다음 페이지가 있으면 `page += 1`과 `time.sleep(0.5)`를 수행하세요.
5. 반복이 정상 종료한 뒤 같은 `try` 블록 안에서 `pd.DataFrame(all_records, columns=COLUMNS)`로 `all_df`를 만드세요. `quotes_raw.csv`에 `index=False`, `encoding="utf-8-sig"`로 저장하세요.
6. 같은 파일을 다시 읽어 `all_loaded_df`에 저장하세요. 마지막 처리 페이지, 최종 `has_next`, 전체 수집 건수, 저장 후 행 수, 컬럼 목록, `head()`를 출력하세요.
7. 교안의 예외 묶음 `(requests.RequestException, ValueError, KeyError, TypeError, RuntimeError)`으로 오류를 처리하세요. 중단 이유와 이번 실행의 저장을 완료하지 못했다는 안내를 출력하세요. `except` 안에서 부분 결과를 CSV로 저장하지 마세요.
8. 실행 결과 확인표와 Q1~Q5에 답하세요. 아래 가정 사례는 실제 사이트 출력과 구분하여 계산하세요.

#### 해석 질문

**Q1.** `page`와 `has_next`는 각각 어떤 역할인가요? 마지막 페이지에서 데이터를 추가한 뒤 `break`를 실행하는 이유는 무엇인가요?

**Q2.** 명언 개수·has_next가 1페이지 10개·True, 2페이지 10개·True, 3페이지 5개·False라면 요청 페이지 수와 최종 건수는 얼마인가요?

**Q3.** `all_records = []`를 반복문 밖에서 만드는 이유는 무엇인가요? `max_pages=20`과 `time.sleep(0.5)`는 각각 무엇을 위한 설정인가요?

**Q4.** 3페이지에서 HTTP 오류가 나거나 명언 목록이 비어 있으면 어떻게 처리하나요? 앞선 페이지의 데이터와 이전 실행의 CSV가 있어도 완료라고 할 수 있나요?

**Q5.** 정상 종료 후 저장 결과에서 무엇을 확인해야 하나요? CSV 저장 성공이 중복·결측 등 데이터 품질 검증의 완료를 뜻하나요?

#### 제출 결과

- 페이지별 반복 수집 코드와 진행 출력
- 성공한 경우 quotes_raw.csv 및 다시 읽은 결과
- 실행 결과 확인표와 Q1~Q5 답변


### 작성 도움말 — 필수 2

바깥 while은 페이지를 반복하고, 안쪽 for는 이번 페이지의 명언을 반복합니다. 이 두 단위를 나누어 생각하면 전체 흐름을 읽기 쉽습니다.

- 누적 목록은 while 앞에서 만들고, 반복 안에서는 레코드를 추가합니다.
- 현재 페이지의 명언을 모두 담은 다음 has_next를 확인합니다. 다음 페이지가 없다는 말이 현재 명언도 없다는 뜻은 아닙니다.
- 다음 요청이 필요할 때만 번호를 늘리고 대기합니다. 상한 초과는 정상 완료와 구분하세요.
- 저장 코드는 while 밖이면서 try 안에 둡니다. 수집 중 오류가 나면 저장을 건너뛰어야 합니다.
- 확인표에는 실제 실행 결과를, 10·10·5개 질문에는 가정 사례의 계산을 적습니다. 두 숫자를 섞지 마세요.


### 코드 작성란

아래 셀에 코드를 작성하고 실제 실행 결과를 남기세요.


In [ ]:
# 필수 2. 여러 페이지 수집과 CSV 저장 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.


### 답변 작성란


| 확인 항목 | 실제 실행 결과·오류 상황 | 해석 |
|---|---|---|
| 마지막으로 정상 처리한 페이지 |  |  |
| 정상 종료 시 `has_next` |  |  |
| 누적 수집 건수 |  |  |
| 저장 후 행 수 |  |  |
| 저장된 컬럼 |  |  |
| 저장 파일과 다시 읽기 결과 |  |  |

가정 사례(10 + 10 + 5)의 요청 페이지 수 / 최종 건수:


**Q1. `page`와 `has_next`는 각각 어떤 역할인가요? 마지막 페이지에서 데이터를 추가한 뒤 `break`를 실행하는 이유는 무엇인가요?**

- 답변: 

**Q2. 명언 개수·has_next가 1페이지 10개·True, 2페이지 10개·True, 3페이지 5개·False라면 요청 페이지 수와 최종 건수는 얼마인가요?**

- 답변: 

**Q3. `all_records = []`를 반복문 밖에서 만드는 이유는 무엇인가요? `max_pages=20`과 `time.sleep(0.5)`는 각각 무엇을 위한 설정인가요?**

- 답변: 

**Q4. 3페이지에서 HTTP 오류가 나거나 명언 목록이 비어 있으면 어떻게 처리하나요? 앞선 페이지의 데이터와 이전 실행의 CSV가 있어도 완료라고 할 수 있나요?**

- 답변: 

**Q5. 정상 종료 후 저장 결과에서 무엇을 확인해야 하나요? CSV 저장 성공이 중복·결측 등 데이터 품질 검증의 완료를 뜻하나요?**

- 답변:


---

## 과제 1. 다른 페이지에 API 수집·저장 적용

### 문제 3-1. 두 번째 페이지의 명언만 별도 파일로 저장하기

#### 문제 설명

필수 1의 요청·추출 방법과 필수 2의 저장 방법을 활용하여 2페이지 한 페이지만 수집하세요. 반복 수집을 새로 구현하지 않습니다. 기존 결과 List를 재사용하지 않고 이 과제의 요청 결과를 독립적으로 저장합니다.

#### 요구사항

1. `assignment_records = []`, `assignment_params = {"page": 2}`를 준비하세요.
2. `try` 안에서 `timeout=10`으로 API를 요청해 `assignment_response`에 저장하세요. 요청 URL·상태 코드·응답 형식을 출력하고 `raise_for_status()`와 `.json()`을 수행하세요.
3. 응답을 `assignment_data`, 명언 목록을 `assignment_quotes`에 저장하세요. 필수 2처럼 명언 목록이 List가 아니거나 비어 있으면 ValueError로 중단하세요.
4. 네 공통 필드를 추출하여 `assignment_records`에 추가하세요. 출처에는 `assignment_response.url`을 사용하세요. 응답 페이지·`has_next`·수집 건수를 출력하세요. `has_next=True`여도 이 과제에서는 3페이지로 이동하지 않습니다.
5. `assignment_df`를 컬럼 순서 `COLUMNS`로 만들고 `quotes_page2.csv`에 `index=False`, `encoding="utf-8-sig"`로 저장하세요.
6. 같은 CSV를 다시 읽어 `assignment_loaded_df`에 저장하고 수집 건수·저장 후 행 수·컬럼·`head()`를 출력하세요. 필수 2와 같은 예외 묶음으로 오류를 처리하고, 오류 시 부분 결과를 저장하지 마세요.
7. 실행 결과 확인표와 Q1~Q4에 답하세요. 파일의 수집 범위가 전체가 아니라 2페이지임을 설명하세요.

#### 해석 질문

**Q1.** 첫 번째 페이지 대신 두 번째 페이지를 요청하려면 무엇을 바꾸나요? 변경된 조건은 `source_url`에서 어떻게 확인하나요?

**Q2.** 과제에서 `has_next=True`가 나와도 다음 페이지를 요청하지 않는 이유는 무엇인가요? 필수 2와 성공 판단의 범위가 어떻게 다른가요?

**Q3.** 이 과제의 행 수와 네 컬럼은 무엇으로 결정되나요? 태그가 여러 개면 행도 여러 개가 되나요?

**Q4.** 403·404·429·500 오류 또는 연결 오류·Timeout이 발생하면 각각 무엇부터 확인하겠나요? 이번 과제에서 자동 재시도를 구현해야 하나요?

#### 제출 결과

- 2페이지 요청부터 저장·다시 읽기까지의 코드와 출력
- 성공한 경우 quotes_page2.csv
- 확인표와 Q1~Q4 답변


### 작성 도움말 — 과제 1

필수 1의 한 페이지 요청과 필수 2의 목록 검사·CSV 저장을 연결합니다. 이번 입력 조건은 page=2입니다.

- 과제의 빈 목록에서 시작하고, 과제 응답에서 얻은 값만 담습니다.
- 명언마다 반복하는 for는 필요하지만 다음 페이지를 요청하는 while은 요구하지 않습니다.
- 출력된 has_next 값은 관찰하되, 과제에서 정한 수집 범위를 유지합니다.
- 저장한 파일과 다시 읽은 파일이 모두 quotes_page2.csv인지 확인하세요.
- 결과를 설명할 때 실제 건수와 함께 '2페이지 한 페이지'라는 범위를 적습니다.


### 코드 작성란

아래 셀에 코드를 작성하고 실제 실행 결과를 남기세요.


In [ ]:
# 과제 1. 다른 페이지에 API 수집·저장 적용 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.


### 답변 작성란


| 확인 항목 | 실행 결과·오류 상황 | 해석 |
|---|---|---|
| 요청 URL / 상태 코드 / 응답 형식 |  |  |
| 응답 페이지 / 다음 페이지 존재 |  |  |
| 수집 건수 / 저장 후 행 수 |  |  |
| 컬럼 목록 |  |  |
| 저장 파일 / 수집 범위 |  |  |


**Q1. 첫 번째 페이지 대신 두 번째 페이지를 요청하려면 무엇을 바꾸나요? 변경된 조건은 `source_url`에서 어떻게 확인하나요?**

- 답변: 

**Q2. 과제에서 `has_next=True`가 나와도 다음 페이지를 요청하지 않는 이유는 무엇인가요? 필수 2와 성공 판단의 범위가 어떻게 다른가요?**

- 답변: 

**Q3. 이 과제의 행 수와 네 컬럼은 무엇으로 결정되나요? 태그가 여러 개면 행도 여러 개가 되나요?**

- 답변: 

**Q4. 403·404·429·500 오류 또는 연결 오류·Timeout이 발생하면 각각 무엇부터 확인하겠나요? 이번 과제에서 자동 재시도를 구현해야 하나요?**

- 답변:


---

## 실습 마무리

**Q1. 이번 실습에서 해결하려던 수집 문제와 전체 처리 흐름은 무엇인가요?**

- 답변: 

**Q2. API 요청 성공, JSON 파싱 성공, 명언 추출 성공은 어떻게 다른가요?**

- 답변: 

**Q3. 필수 2가 마지막 페이지까지 완료되었다고 판단할 근거는 무엇인가요?**

- 답변: 

**Q4. 필수 2에서 20페이지 상한 때문에 중단되었다면 정상 완료라고 할 수 있나요?**

- 답변: 

**Q5. CSV를 다시 읽을 때 확인한 항목은 무엇이며, 이후 품질 검증과 어떻게 구분하나요?**

- 답변: 

**Q6. 필수 2와 과제 1의 파일이 각각 어떤 수집 범위를 담는지 설명하세요.**

- 답변:


### 제출 전 확인

- 필수 1·2와 과제 1의 코드·실행 결과·확인표·질문 답변을 포함하세요.
- 실습 마무리 질문까지 답하세요.
- 실제 실행 결과와 문제에서 가정한 숫자를 구분하세요.
- 오류가 났다면 오류 상황을 기록하고, 이전 CSV를 이번 성공 결과로 제출하지 마세요.
- 정상 저장한 `quotes_raw.csv`, `quotes_page2.csv`와 작성한 `.ipynb`를 제출하세요.


## 개념과 결과를 함께 정리하기

| 구분 | 이번 강의에서의 의미 |
|---|---|
| API 주소 / 요청 조건 | 어느 기능을 사용할지 / 그 기능에 어떤 페이지를 요청할지 |
| 응답 객체 / 파싱한 값 | 상태·헤더·본문을 가진 응답 / JSON을 해석한 dict·list 등 |
| 요청 성공 / 추출 성공 | HTTP 처리 성공 / 필요한 값으로 레코드 구성 성공 |
| 현재 페이지 건수 / 누적 건수 | 이번 quotes의 길이 / all_records에 모인 전체 길이 |
| 다음 페이지 없음 / 명언 목록 없음 | 정상 종료 판단 정보 / 전체 수집 예제에서 중단할 목록 문제 |
| break / 예외 | 정상적으로 반복을 끝내 저장으로 진행 / 지정한 오류 처리로 이동 |
| 최대 페이지 / 마지막 페이지 | 코드가 정한 상한 / API의 has_next로 판단한 종료 위치 |
| timeout / sleep | 통신 대기 제한 / 다음 요청 전 의도적인 간격 |
| raw CSV / 정제 완료 데이터 | 기본 수집·저장 결과 / 이후 품질 확인·정제가 필요한 결과 |

### 막혔을 때 확인할 순서

1. **준비:** API_URL·COLUMNS와 라이브러리를 준비했는지 확인합니다.
2. **요청:** 요청 조건, 응답 URL과 상태 코드 또는 연결 오류 메시지를 확인합니다.
3. **파싱:** 응답을 JSON으로 해석할 수 있었는지 확인합니다.
4. **추출:** quotes 목록과 명언 내부의 Key를 확인합니다.
5. **반복:** 현재 page, 이번 건수, 누적 건수, has_next를 함께 읽습니다.
6. **저장:** 수집의 정상 종료 여부와 현재 실행의 저장·다시 읽기 출력을 확인합니다.

오류 없이 실행됐다는 말만으로 결과를 설명하지 말고, **어느 페이지 범위에서 무엇을 몇 개 얻었고, 어떤 근거로 완료라고 판단했는지** 자신의 말로 설명해 보세요. 교안의 요청·파싱·반복·기본 저장 확인 범위 안에서 정리하면 됩니다.
